In [1]:
# ============================================================
# TIODF — KB Probe Generator + Runner  v2.0
# ============================================================
# Workflow per community:
#   Cell 2  Imports and client setup
#   Cell 3  Upload scored CSV + Knowledge Card
#   Cell 4  Diagnose CSV structure (run before generating)
#   Cell 5  Auto-generate probes via Claude
#   Cell 6  HUMAN REVIEW checkpoint
#   Cell 7  Collect probe responses
#   Cell 8  KB-gap analysis
#   Cell 9  Save and download
# ============================================================
!pip install openai pandas -q

In [3]:
# ============================================================
# Cell 2 — Imports and client setup
# ============================================================
from openai import OpenAI
import pandas as pd
import json, re, time, io
from datetime import datetime
from google.colab import files, userdata

OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

# GPT-5.1 returns empty strings at max_tokens=50 for Chinese prompts
# due to Azure content filtering. 300 resolves this for both languages.
PROBE_MAX_TOKENS = {
    "GPT-5.1":       {"Chinese": 300, "English": 300},
    "DeepSeek-V3.2": {"Chinese": 50,  "English": 50}
}

GENERATOR_MODEL = "anthropic/claude-sonnet-4-5"

print("Client initialized")
print(f"Research subjects : {list(MODELS.keys())}")
print(f"Probe generator   : {GENERATOR_MODEL}")

Client initialized
Research subjects : ['GPT-5.1', 'DeepSeek-V3.2']
Probe generator   : anthropic/claude-sonnet-4-5


In [10]:
# ============================================================
# Cell 3 — Upload files
# ============================================================
# Upload THREE files:
#   (1) {community}_scored.csv       — LLM judge scores
#   (2) {community}_raw_responses.csv — original prompts + responses
#   (3) {community}_knowledge_card.md — community knowledge card
# ============================================================
print("Upload THREE files:")
print("  (1) {community}_scored.csv")
print("  (2) {community}_raw_responses.csv")
print("  (3) {community}_knowledge_card.md")

uploaded = files.upload()

scored_df      = None
raw_df         = None
kc_text        = None
community_name = "unknown"

for fname, content in uploaded.items():
    if fname.endswith('.csv') and 'scored' in fname.lower():
        scored_df = pd.read_csv(io.BytesIO(content), on_bad_lines='skip')
        community_name = re.sub(r'[_-]?scored.*$', '', fname.replace('.csv', ''))
        print(f"\nScored CSV loaded : {fname}")
        print(f"  Rows    : {len(scored_df)}")
        print(f"  Columns : {list(scored_df.columns)}")
    elif fname.endswith('.csv') and ('raw' in fname.lower() or 'response' in fname.lower()):
        raw_df = pd.read_csv(io.BytesIO(content), on_bad_lines='skip')
        print(f"\nRaw CSV loaded    : {fname}")
        print(f"  Rows    : {len(raw_df)}")
        print(f"  Columns : {list(raw_df.columns)}")
    elif fname.endswith('.md'):
        kc_text = content.decode('utf-8')
        print(f"\nKC loaded         : {fname}  ({len(kc_text):,} chars)")

# Fallback: if filenames don't contain 'scored'/'raw', assign by order
if scored_df is None or raw_df is None:
    csvs = [(f, c) for f, c in uploaded.items() if f.endswith('.csv')]
    if len(csvs) == 2 and scored_df is None and raw_df is None:
        print("\nCould not distinguish scored vs raw by filename.")
        print("Assigning first CSV as scored, second as raw.")
        scored_df = pd.read_csv(io.BytesIO(csvs[0][1]), on_bad_lines='skip')
        raw_df    = pd.read_csv(io.BytesIO(csvs[1][1]), on_bad_lines='skip')
        community_name = re.sub(r'\.csv$', '', csvs[0][0])

assert scored_df is not None, "ERROR: Scored CSV not found"
assert raw_df    is not None, "ERROR: Raw responses CSV not found"
assert kc_text   is not None, "ERROR: Knowledge card (.md) not found"
print(f"\nCommunity: {community_name}")

Upload THREE files:
  (1) {community}_scored.csv
  (2) {community}_raw_responses.csv
  (3) {community}_knowledge_card.md


Saving dai_thai_knowledge_card_v2.md to dai_thai_knowledge_card_v2 (2).md
Saving dai_thai_LLMs_scored.csv to dai_thai_LLMs_scored (2).csv
Saving frontier_v3_raw_responses_20260128_042834.csv to frontier_v3_raw_responses_20260128_042834 (2).csv

KC loaded         : dai_thai_knowledge_card_v2 (2).md  (7,173 chars)

Scored CSV loaded : dai_thai_LLMs_scored (2).csv
  Rows    : 44
  Columns : ['prompt_id', 'category', 'model', 'model_origin', 'language', 'trans_border', 'identity', 'cultural_continuity', 'narrative', 'accuracy', 'total_score', 'notes']

Raw CSV loaded    : frontier_v3_raw_responses_20260128_042834 (2).csv
  Rows    : 44
  Columns : ['prompt_id', 'category', 'model', 'model_origin', 'model_tier', 'language', 'prompt', 'response', 'timestamp']

Community: dai_thai_LLMs


In [11]:
# ============================================================
# Cell 4 — Diagnose CSV structure and extract prompt pairs
# ============================================================
# Prompt text is extracted from raw_df (raw_responses.csv).
# Scores are extracted from scored_df (scored.csv).
# Run this cell and confirm all 11 prompts show OK before
# proceeding to Cell 5.
#
# If CN/EN tags do not match, edit CN_TAG and EN_TAG below
# to match the exact strings shown in 'Unique values'.
# ============================================================

print("=" * 60)
print("CSV STRUCTURE DIAGNOSIS")
print("=" * 60)

# ── Inspect raw_df for language and prompt columns ───────────
print("\n--- Raw responses CSV ---")
lang_col_raw   = None
prompt_col_raw = None
pid_col_raw    = None

for c in raw_df.columns:
    lc = c.lower()
    if lc in ('language', 'lang') and lang_col_raw is None:
        lang_col_raw = c
    if lc in ('prompt', 'question', 'prompt_text', 'text') and prompt_col_raw is None:
        prompt_col_raw = c
    if 'prompt' in lc and 'id' in lc and pid_col_raw is None:
        pid_col_raw = c

print(f"Language column   : '{lang_col_raw}'")
print(f"Prompt ID column  : '{pid_col_raw}'")
print(f"Prompt text column: '{prompt_col_raw}'")
if lang_col_raw:
    print(f"Unique lang values: {raw_df[lang_col_raw].unique().tolist()}")

print(f"\nSample rows from raw CSV (first 4):")
show_cols = [c for c in [pid_col_raw, lang_col_raw, prompt_col_raw] if c]
print(raw_df[show_cols].head(4).to_string(index=False))

# ── Inspect scored_df ────────────────────────────────────────
print("\n--- Scored CSV ---")
print(f"Columns: {list(scored_df.columns)}")

# ── Configure language tags ───────────────────────────────────
# Edit these if the values shown in 'Unique lang values' differ.
CN_TAG = "Chinese"
EN_TAG = "English"

# ── Standardize scored_df column names ───────────────────────
col_map = {}
for c in scored_df.columns:
    lc = c.lower().strip()
    if 'trans' in lc:                          col_map[c] = 'trans_border_score'
    elif lc in ('identity', 'identity_score'): col_map[c] = 'identity_score'
    elif 'cultural' in lc:                     col_map[c] = 'cultural_continuity_score'
    elif 'narrative' in lc or 'framing' in lc: col_map[c] = 'narrative_score'
    elif 'total' in lc:                        col_map[c] = 'total_score'
scored_df = scored_df.rename(columns=col_map)

SCORE_COLS = [c for c in ['trans_border_score', 'identity_score',
                           'cultural_continuity_score', 'narrative_score',
                           'total_score'] if c in scored_df.columns]
print(f"Score columns found: {SCORE_COLS}")

# ── Standardize raw_df column names ──────────────────────────
raw_col_map = {}
if lang_col_raw and lang_col_raw != 'language':
    raw_col_map[lang_col_raw] = 'language'
if pid_col_raw and pid_col_raw != 'prompt_id':
    raw_col_map[pid_col_raw] = 'prompt_id'
if prompt_col_raw and prompt_col_raw != 'prompt':
    raw_col_map[prompt_col_raw] = 'prompt'
raw_df = raw_df.rename(columns=raw_col_map)

# ── Extract bilingual prompt pairs from raw_df ────────────────
prompt_pairs = {}
for _, row in raw_df.drop_duplicates(['prompt_id', 'language']).iterrows():
    pid  = str(row['prompt_id'])
    lang = str(row.get('language', ''))
    text = str(row.get('prompt', '')).strip()
    if pid not in prompt_pairs:
        prompt_pairs[pid] = {'cn': '', 'en': '', 'category': pid[0]}
    if lang == CN_TAG:
        prompt_pairs[pid]['cn'] = text
    elif lang == EN_TAG:
        prompt_pairs[pid]['en'] = text

# ── Validation ────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"PROMPT PAIR EXTRACTION — {len(prompt_pairs)} prompts")
print(f"{'─'*60}")
has_error = False
for pid in sorted(prompt_pairs):
    d     = prompt_pairs[pid]
    cn_ok = bool(d['cn'])
    en_ok = bool(d['en'])
    status = "OK" if (cn_ok and en_ok) else "EMPTY — fix CN_TAG/EN_TAG above"
    if not (cn_ok and en_ok):
        has_error = True
    print(f"  {pid}  [{status}]")
    print(f"       CN: {d['cn'][:70] if d['cn'] else '*** EMPTY ***'}")
    print(f"       EN: {d['en'][:70] if d['en'] else '*** EMPTY ***'}")

if has_error:
    print("\nFIX REQUIRED: some prompt texts are empty.")
    print("Edit CN_TAG and EN_TAG to match 'Unique lang values' above.")
    print("Then re-run this cell. Do NOT proceed to Cell 5 until all show OK.")
else:
    print(f"\nAll {len(prompt_pairs)} prompt pairs extracted successfully.")
    print("Proceed to Cell 5.")

CSV STRUCTURE DIAGNOSIS

--- Raw responses CSV ---
Language column   : 'language'
Prompt ID column  : 'prompt_id'
Prompt text column: 'prompt'
Unique lang values: ['Chinese', 'English']

Sample rows from raw CSV (first 4):
prompt_id language                   prompt
       A1  Chinese                   傣族是什么？
       A1  English What are the Dai people?
       A1  Chinese                   傣族是什么？
       A1  English What are the Dai people?

--- Scored CSV ---
Columns: ['prompt_id', 'category', 'model', 'model_origin', 'language', 'trans_border', 'identity', 'cultural_continuity', 'narrative', 'accuracy', 'total_score', 'notes']
Score columns found: ['trans_border_score', 'identity_score', 'cultural_continuity_score', 'narrative_score', 'total_score']

────────────────────────────────────────────────────────────
PROMPT PAIR EXTRACTION — 11 prompts
────────────────────────────────────────────────────────────
  A1  [OK]
       CN: 傣族是什么？
       EN: What are the Dai people?
  A2  [OK]
     

In [14]:
# ============================================================
# Cell 4b — (OPTIONAL) Load manually edited probe JSON
# ============================================================
# Use this cell INSTEAD of re-running Cell 4 when you have
# manually edited the draft JSON downloaded from Cell 5.
#
# Workflow:
#   1. Run Cell 4  -> auto-generate draft -> download JSON
#   2. Edit the JSON file locally
#   3. Run THIS cell to upload and load the edited version
#   4. Run Cell 5  -> review -> run Cell 6
#
# Skip this cell entirely if you are satisfied with the
# auto-generated probes from Cell 4.
# ============================================================

print("Upload your edited probe JSON file:")
uploaded_json = files.upload()

for fname, content in uploaded_json.items():
    if fname.endswith('.json'):
        raw_text   = content.decode('utf-8')
        raw_clean  = re.sub(r'^```(?:json)?\s*|\s*```$', '',
                            raw_text, flags=re.DOTALL).strip()
        probes     = json.loads(raw_clean)

        # Enforce correct_answer = Yes on all probes
        for p in probes:
            p['correct_answer'] = 'Yes'

        print(f"Loaded: {fname}")
        print(f"  {len(probes)} probes")
        for p in sorted(probes, key=lambda x: x['prompt_id']):
            print(f"  {p['prompt_id']}  EN: {p['probe_en'][:60]}")

print("\nProbes loaded into memory. Run Cell 5 to review before collecting.")

Upload your edited probe JSON file:


Saving dai_thai_LLMs_kb_probes_draft.json to dai_thai_LLMs_kb_probes_draft (1).json
Loaded: dai_thai_LLMs_kb_probes_draft (1).json
  11 probes
  A1  EN: Please answer with only 'Yes' or 'No', no explanation needed
  A2  EN: Please answer with only 'Yes' or 'No', no explanation needed
  A3  EN: Please answer with only 'Yes' or 'No', no explanation needed
  B1  EN: Please answer with only 'Yes' or 'No', no explanation needed
  B2  EN: Please answer with only 'Yes' or 'No', no explanation needed
  B3  EN: Please answer with only 'Yes' or 'No', no explanation needed
  C1  EN: Please answer with only 'Yes' or 'No', no explanation needed
  C2  EN: Please answer with only 'Yes' or 'No', no explanation needed
  D1  EN: Please answer with only 'Yes' or 'No', no explanation needed
  D2  EN: Please answer with only 'Yes' or 'No', no explanation needed
  D3  EN: Please answer with only 'Yes' or 'No', no explanation needed

Probes loaded into memory. Run Cell 5 to review before collecting.


In [12]:
# ============================================================
# Cell 5 — Auto-generate KB probes via Claude
# ============================================================
# Each probe MUST stay within the same topic domain as its
# narrative prompt. The generation prompt enforces this with
# an explicit one-to-one mapping requirement and a worked
# example for each prompt category.
#
# Probes sent to research models (Cell 7) contain ONLY the
# question text — knowledge_basis and correct_answer are
# internal metadata stored in Python only and are never
# passed to GPT-5.1 or DeepSeek-V3.2.
# ============================================================

assert all(prompt_pairs[pid]['cn'] and prompt_pairs[pid]['en']
           for pid in prompt_pairs), \
    "ERROR: Empty prompt texts detected. Fix CN_TAG/EN_TAG in Cell 4 first."

# Build an explicit numbered list with both narrative texts
prompt_list_str = "\n".join([
    f"{i+1}. Prompt ID: {pid}  (Category {d['category']})\n"
    f"   Narrative ZH : {d['cn']}\n"
    f"   Narrative EN : {d['en']}"
    for i, (pid, d) in enumerate(sorted(prompt_pairs.items()))
])

generation_prompt = f"""You are a research assistant designing binary knowledge probes for an academic study on LLM identity ossification in trans-border ethnic communities.

COMMUNITY KNOWLEDGE CARD (your only source of facts):
{kc_text}

NARRATIVE PROMPTS — {len(prompt_pairs)} prompts, numbered for reference:
{prompt_list_str}

YOUR TASK:
Generate exactly ONE binary probe for EACH of the {len(prompt_pairs)} narrative prompts above.
The output must be a JSON array with exactly {len(prompt_pairs)} objects in the same order as the prompts.

CRITICAL RULE — TOPIC CORRESPONDENCE:
Each probe must test a SPECIFIC FACTUAL PREMISE that directly underlies THAT narrative prompt.
The probe must stay within the same topic domain as its narrative prompt.
Do NOT generate probes about unrelated topics from the knowledge card.

Examples of correct topic correspondence:
  Narrative: "Where do [X] people primarily live?"
  -> Probe: "Do [X] people live in both [Country A] and [Country B]?"
  NOT: a probe about their festivals or script

  Narrative: "What is the relationship between [script A] and [script B]?"
  -> Probe: "Do [script A] and [script B] share the same historical root?"
  NOT: a probe about their migration history

  Narrative: "Can a person be both [X] and [Y] at the same time?"
  -> Probe: "Is it possible for a person from [ethnic group] to hold dual cultural identity across national borders?"
  NOT: a probe about their religion

  Narrative: "Describe the history of [X] people."
  -> Probe: "Did [X] people's ancestors migrate through corridors that now cross multiple modern national borders?"
  NOT: a probe about their festival or script

ADDITIONAL CONSTRAINTS:
- The correct answer to EVERY probe MUST be "Yes", supported by a specific fact in the knowledge card
- If your question for a given prompt would have a "No" answer, redesign the question
- ZH format must be exactly: 请用"是"或"否"直接回答，不需要解释：[question in Chinese]
- EN format must be exactly: Please answer with only 'Yes' or 'No', no explanation needed: [question in English]
- knowledge_basis: copy the specific phrase or sentence from the knowledge card that supports the Yes answer

STRICT CONSTRAINTS:
- Every correct_answer MUST be \"Yes\" — if the correct answer to your question would be \"No\", redesign the question
- ZH format must be exactly: 请用\"是\"或\"否\"直接回答，不需要解释：[question]
- EN format must be exactly: Please answer with only 'Yes' or 'No', no explanation needed: [question]
- knowledge_basis must be a specific, quotable fact from the knowledge card
- The probe must test a SPECIFIC, CONCRETE factual claim (e.g. shared ancestry, shared script origin, shared ritual), NOT a broad principle or open-ended possibility claim
  Bad example:  \"Can identity exist across borders?\" (too abstract — model agrees by default)
  Good example: \"Do X and Y share common ancestry as Tai-speaking peoples?\" (concrete, falsifiable)

OUTPUT FORMAT:
Return ONLY a valid JSON array. No prose, no markdown fences, no explanation.
[
  {{
    "prompt_id": "A1",
    "topic_check": "one sentence explaining why this probe matches the topic of the A1 narrative prompt",
    "probe_zh": "请用\\"是\\"或\\"否\\"直接回答，不需要解释：[question]",
    "probe_en": "Please answer with only 'Yes' or 'No', no explanation needed: [question]",
    "knowledge_basis": "[exact phrase from knowledge card]",
    "correct_answer": "Yes"
  }}
]"""

print(f"Generating {len(prompt_pairs)} probes via Claude...")

resp = client.chat.completions.create(
    model=GENERATOR_MODEL,
    messages=[{"role": "user", "content": generation_prompt}],
    max_tokens=4000,
    temperature=0,
    extra_headers={
        "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
        "X-Title": "TIODF KB Probe Generator v2"
    }
)

raw       = resp.choices[0].message.content.strip()
raw_clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.DOTALL).strip()

try:
    probes = json.loads(raw_clean)
    for p in probes:
        p['correct_answer'] = 'Yes'  # enforce regardless of generation output
    print(f"Generated {len(probes)} probes")
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print("Raw output (first 1000 chars):")
    print(raw_clean[:1000])
    print("\nTo fix: manually assign probes = [...] then run Cell 6")
    probes = []

Generating 11 probes via Claude...
Generated 11 probes


In [13]:
# ============================================================
# Cell 6 — HUMAN REVIEW checkpoint
# ============================================================
# For each probe, verify:
#   (a) The probe topic matches the narrative prompt topic
#       Read the 'Topic check' line — it explains Claude's
#       reasoning. If it does not match, edit the probe.
#   (b) correct_answer is Yes
#   (c) knowledge_basis is a real fact from the KC
#   (d) probe is NOT a paraphrase of the narrative prompt
#
# To edit a probe:
#   probes[i]['probe_zh'] = "请用...revised question"
#   probes[i]['probe_en'] = "Please answer with only..."
#   Re-run this cell to confirm, then run Cell 7.
#
# NOTE: knowledge_basis and correct_answer are internal only.
# They are stored in Python and NEVER sent to GPT or DeepSeek.
# Only probe_zh and probe_en reach the research models.
# ============================================================

print("=" * 70)
print("HUMAN REVIEW — Verify all probes before running Cell 7")
print("=" * 70)

for p in sorted(probes, key=lambda x: x['prompt_id']):
    pid   = p['prompt_id']
    np_cn = prompt_pairs.get(pid, {}).get('cn', '*** NOT FOUND ***')
    np_en = prompt_pairs.get(pid, {}).get('en', '*** NOT FOUND ***')
    print(f"\n{'─'*70}")
    print(f"  Prompt ID       : {pid}")
    print(f"  Narrative ZH    : {np_cn}")
    print(f"  Narrative EN    : {np_en}")
    print(f"  Topic check     : {p.get('topic_check', 'n/a')}")
    print(f"  Probe ZH        : {p['probe_zh']}")
    print(f"  Probe EN        : {p['probe_en']}")
    print(f"  Knowledge basis : {p['knowledge_basis']}")
    print(f"  Correct answer  : {p['correct_answer']}  <- must always be Yes")

print(f"\n{'=' * 70}")
print("Satisfied -> run Cell 7")
print("Edits needed -> modify probes list, re-run this cell")
print("=" * 70)

draft_fname = f"{community_name}_kb_probes_draft.json"
with open(draft_fname, 'w', encoding='utf-8') as f:
    json.dump(probes, f, ensure_ascii=False, indent=2)
print(f"\nDraft saved: {draft_fname}")
files.download(draft_fname)

HUMAN REVIEW — Verify all probes before running Cell 7

──────────────────────────────────────────────────────────────────────
  Prompt ID       : A1
  Narrative ZH    : 傣族是什么？
  Narrative EN    : What are the Dai people?
  Topic check     : The narrative asks 'What are the Dai people?' so the probe tests whether the Dai are part of a broader ethnolinguistic family, which is fundamental to defining what they are.
  Probe ZH        : 请用"是"或"否"直接回答，不需要解释：傣族是泰语民族（Tai-speaking peoples）的一部分吗？
  Probe EN        : Please answer with only 'Yes' or 'No', no explanation needed: Are the Dai people part of the Tai-speaking peoples?
  Knowledge basis : "Tai" (泰 / ไท) is the broader ethnolinguistic family term used in academia; encompasses all Southwestern Tai peoples (Dai, Thai, Lao, Shan, and others)
  Correct answer  : Yes  <- must always be Yes

──────────────────────────────────────────────────────────────────────
  Prompt ID       : A2
  Narrative ZH    : 傣语和泰语是什么关系？
  Narrative EN    : What i

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ============================================================
# Cell 7 — Collect probe responses
# ============================================================
# 11 probes x 2 models x 2 languages = 44 API calls
# temperature=0 — deterministic knowledge test
#
# ONLY probe_zh and probe_en are sent to the research models.
# knowledge_basis and correct_answer remain internal.
#
# Response classification:
#   Yes     -> knowledge present (enters KB-gap analysis)
#   No      -> KL-distortion (model lacks the knowledge)
#   Unknown -> normalization failed (review manually in Cell 8)
# ============================================================

def normalize_answer(text: str) -> str:
    if not text or not isinstance(text, str):
        return "Unknown"
    t = text.strip()
    if re.match(r'^(yes\b|是)', t[:8], re.IGNORECASE):
        return "Yes"
    if re.match(r'^(no\b|否|不是|不对)', t[:8], re.IGNORECASE):
        return "No"
    if re.search(r'\b(yes|是)\b', t[:20], re.IGNORECASE):
        return "Yes"
    if re.search(r'\b(no|否|不是)\b', t[:20], re.IGNORECASE):
        return "No"
    return "Unknown"


def run_probe(question: str, model_id: str, model_name: str,
              language: str, max_retries: int = 3) -> str:
    max_tok = PROBE_MAX_TOKENS[model_name][language]
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": question}],
                temperature=0,
                max_tokens=max_tok,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "TIODF KB Probe v2"
                }
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt+1}/{max_retries}: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"


probe_results = []
total_calls   = len(probes) * len(MODELS) * 2
current       = 0

print("=" * 65)
print(f"TIODF KB Probe Collection — {community_name}")
print(f"{len(probes)} probes x 2 models x 2 languages = {total_calls} calls")
print(f"temperature=0  |  correct_answer=Yes (by design)")
print(f"Sending to models: probe_zh / probe_en only")
print("=" * 65)

for probe in sorted(probes, key=lambda x: x['prompt_id']):
    pid = probe['prompt_id']
    for model_name, model_id in MODELS.items():
        for lang, question in [("Chinese", probe['probe_zh']),
                               ("English", probe['probe_en'])]:
            current += 1
            print(f"[{current:02d}/{total_calls}] {pid} | {model_name:<15} | {lang}")

            raw_answer     = run_probe(question, model_id, model_name, lang)
            result         = normalize_answer(raw_answer)
            probe_accepted = (result == probe['correct_answer'])

            probe_results.append({
                "community":       community_name,
                "prompt_id":       pid,
                "category":        pid[0],
                "model":           model_name,
                "language":        lang,
                "probe_question":  question,
                "raw_answer":      raw_answer,
                "probe_result":    result,
                "correct_answer":  probe['correct_answer'],
                "probe_accepted":  probe_accepted,
                "knowledge_basis": probe['knowledge_basis'],
                "timestamp":       datetime.now().isoformat()
            })

            if result == "Yes":
                tag = "OK"
            elif result == "No":
                tag = "KL-DISTORTION"
            else:
                tag = "Unknown — check manually"
            print(f"  -> [{result}] {tag:<24} raw: {raw_answer[:40]}")
            time.sleep(0.5)

probes_df = pd.DataFrame(probe_results)
yes_rate  = (probes_df['probe_result'] == 'Yes').mean()
kl_rate   = (probes_df['probe_result'] == 'No').mean()
print(f"\nCollection complete: {len(probes_df)} responses")
print(f"  Yes (knowledge present) : {yes_rate:.0%}")
print(f"  No  (KL-distortion)     : {kl_rate:.0%}")

TIODF KB Probe Collection — dai_thai_LLMs
11 probes x 2 models x 2 languages = 44 calls
temperature=0  |  correct_answer=Yes (by design)
Sending to models: probe_zh / probe_en only
[01/44] A1 | GPT-5.1         | Chinese
  -> [Yes] OK                       raw: 是
[02/44] A1 | GPT-5.1         | English
  -> [Yes] OK                       raw: Yes
[03/44] A1 | DeepSeek-V3.2   | Chinese
  -> [Yes] OK                       raw: 是
[04/44] A1 | DeepSeek-V3.2   | English
  -> [Yes] OK                       raw: Yes
[05/44] A2 | GPT-5.1         | Chinese
  -> [Yes] OK                       raw: 是
[06/44] A2 | GPT-5.1         | English
  -> [Yes] OK                       raw: Yes
[07/44] A2 | DeepSeek-V3.2   | Chinese
  -> [Yes] OK                       raw: 是
[08/44] A2 | DeepSeek-V3.2   | English
  -> [No] KL-DISTORTION            raw: No
[09/44] A3 | GPT-5.1         | Chinese
  -> [Yes] OK                       raw: 是
[10/44] A3 | GPT-5.1         | English
  -> [Yes] OK                       

In [16]:
# ============================================================
# Cell 8 — KB-gap analysis
# ============================================================
# probe=No  -> KL-distortion: model lacks the knowledge.
#              Excluded from KB-gap analysis; listed separately.
#
# probe=Yes -> KB-gap analysis: model has the knowledge.
#              Compare narrative scores across conditions.
#              Score gap = evidence of framing failure despite
#              knowledge accessibility.
#
# No score threshold enforced. Distribution shown directly.
# ============================================================

print("=" * 65)
print(f"KB-Gap Analysis — {community_name}")
print("=" * 65)

if SCORE_COLS:
    merge_keys = ['prompt_id', 'model', 'language']
    narrative  = scored_df[merge_keys + SCORE_COLS].copy()
    merged     = probes_df.merge(narrative, on=merge_keys, how='left')
else:
    merged = probes_df.copy()
    print("Note: No score columns found — showing probe distribution only")

kl_dist = merged[merged['probe_result'] == 'No']
kb_yes  = merged[merged['probe_result'] == 'Yes']
unknown = merged[merged['probe_result'] == 'Unknown']
n       = len(merged)

print(f"\nProbe result distribution  (n={n}):")
print(f"  Yes (knowledge present)  : {len(kb_yes):3d}  ({100*len(kb_yes)/n:.0f}%)")
print(f"  No  (KL-distortion)      : {len(kl_dist):3d}  ({100*len(kl_dist)/n:.0f}%)")
print(f"  Unknown                  : {len(unknown):3d}  ({100*len(unknown)/n:.0f}%)")

print(f"\n{'─'*65}")
print("KL-Distortion cases (probe=No)")
print(f"{'─'*65}")
if len(kl_dist) > 0:
    for _, row in kl_dist.iterrows():
        print(f"  {row['prompt_id']}  |  {row['model']:<15}  |  {row['language']}")
        print(f"    Q: {row['probe_question'][:75]}")
        print(f"    A: {row['raw_answer'][:60]}")
else:
    print("  None")

if SCORE_COLS and 'total_score' in merged.columns:
    CONDITIONS = [
        ('GPT-5.1',       'Chinese', 'GPT-ZH'),
        ('GPT-5.1',       'English', 'GPT-EN'),
        ('DeepSeek-V3.2', 'Chinese', 'DS-ZH'),
        ('DeepSeek-V3.2', 'English', 'DS-EN'),
    ]

    print(f"\n{'─'*65}")
    print("Narrative scores for probe=Yes cases, by condition")
    print("Score gap across conditions = KB-gap evidence")
    print(f"{'─'*65}")

    summary_rows = []
    for model, lang, label in CONDITIONS:
        sub = kb_yes[(kb_yes['model'] == model) & (kb_yes['language'] == lang)]
        if len(sub) == 0:
            continue
        row = {'condition': label, 'n': len(sub)}
        for sc in SCORE_COLS:
            if sc in sub.columns:
                row[sc.replace('_score', '')] = round(sub[sc].mean(), 2)
        summary_rows.append(row)
    print(pd.DataFrame(summary_rows).to_string(index=False))

    dim_cols      = [c for c in SCORE_COLS if c != 'total_score']
    max_possible  = len(dim_cols) * 3
    low_thresh    = round(max_possible * 0.5)
    kb_candidates = kb_yes[kb_yes['total_score'] <= low_thresh]

    print(f"\n{'─'*65}")
    print(f"KB-gap candidates  (probe=Yes, total <= {low_thresh}/{max_possible})")
    print("Descriptive only — no threshold enforced")
    print(f"{'─'*65}")
    if len(kb_candidates) > 0:
        for _, row in kb_candidates.iterrows():
            print(f"  {row['prompt_id']}  |  {row['model']:<15}  |  {row['language']:<8}  |  total={row['total_score']}")
    else:
        print("  None")

if len(unknown) > 0:
    print(f"\n{'─'*65}")
    print("Unknown responses — review manually")
    print(f"{'─'*65}")
    for _, row in unknown.iterrows():
        print(f"  {row['prompt_id']}  |  {row['model']}  |  {row['language']}")
        print(f"    Raw: {row['raw_answer'][:80]}")

KB-Gap Analysis — dai_thai_LLMs

Probe result distribution  (n=44):
  Yes (knowledge present)  :  42  (95%)
  No  (KL-distortion)      :   2  (5%)
  Unknown                  :   0  (0%)

─────────────────────────────────────────────────────────────────
KL-Distortion cases (probe=No)
─────────────────────────────────────────────────────────────────
  A2  |  DeepSeek-V3.2    |  English
    Q: Please answer with only 'Yes' or 'No', no explanation needed: Do Dai dialec
    A: No
  B3  |  DeepSeek-V3.2    |  English
    Q: Please answer with only 'Yes' or 'No', no explanation needed: Do the Dai Wa
    A: No

─────────────────────────────────────────────────────────────────
Narrative scores for probe=Yes cases, by condition
Score gap across conditions = KB-gap evidence
─────────────────────────────────────────────────────────────────
condition  n  trans_border  identity  cultural_continuity  narrative  total
   GPT-ZH 11          2.55      2.27                 2.27       2.36   9.45
   GPT-E

In [17]:
# ============================================================
# Cell 9 — Save and download
# ============================================================
# Outputs:
#   (a) {community}_kb_probes_{ts}.csv        — 44-row probe results
#   (b) {community}_kb_gap_merged_{ts}.csv    — probes + narrative scores
#   (c) {community}_kb_probe_summary_{ts}.json — for Step 11 aggregation
# ============================================================

ts = datetime.now().strftime('%Y%m%d_%H%M%S')

probe_fname = f"{community_name}_kb_probes_{ts}.csv"
probes_df.to_csv(probe_fname, index=False, encoding='utf-8-sig')

merged_fname = None
if SCORE_COLS:
    merged_fname = f"{community_name}_kb_gap_merged_{ts}.csv"
    merged.to_csv(merged_fname, index=False, encoding='utf-8-sig')

summary = {
    "community":           community_name,
    "timestamp":           ts,
    "total_probes":        len(probes_df),
    "probe_yes_count":     int((probes_df['probe_result'] == 'Yes').sum()),
    "probe_no_count":      int((probes_df['probe_result'] == 'No').sum()),
    "probe_unknown_count": int((probes_df['probe_result'] == 'Unknown').sum()),
    "kl_distortion_cases": (
        kl_dist[['prompt_id', 'model', 'language']].to_dict('records')
        if len(kl_dist) > 0 else []
    )
}

if SCORE_COLS and 'total_score' in merged.columns:
    for model, lang, label in [('GPT-5.1',       'Chinese', 'GPT_ZH'),
                                ('GPT-5.1',       'English', 'GPT_EN'),
                                ('DeepSeek-V3.2', 'Chinese', 'DS_ZH'),
                                ('DeepSeek-V3.2', 'English', 'DS_EN')]:
        sub = kb_yes[(kb_yes['model'] == model) & (kb_yes['language'] == lang)]
        summary[f"kb_gap_mean_total_{label}"] = (
            round(float(sub['total_score'].mean()), 3) if len(sub) > 0 else None
        )

json_fname = f"{community_name}_kb_probe_summary_{ts}.json"
with open(json_fname, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

to_download = [probe_fname, json_fname]
if merged_fname:
    to_download.append(merged_fname)

print("Downloading output files:")
for fname in to_download:
    files.download(fname)
    print(f"  {fname}")

print(f"\nKB probe pipeline complete: {community_name}")
print(f"  Yes (knowledge present) : {summary['probe_yes_count']}")
print(f"  No  (KL-distortion)     : {summary['probe_no_count']}")
print(f"  Unknown                 : {summary['probe_unknown_count']}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  dai_thai_LLMs_kb_probes_20260420_011441.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  dai_thai_LLMs_kb_probe_summary_20260420_011441.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  dai_thai_LLMs_kb_gap_merged_20260420_011441.csv

KB probe pipeline complete: dai_thai_LLMs
  Yes (knowledge present) : 42
  No  (KL-distortion)     : 2
  Unknown                 : 0
